# MetaTrader 5 - TPO Market Profile Analysis

This notebook calculates and visualizes a **TPO (Time Price Opportunity) Market Profile** using **MetaTrader 5 (MT5)** tick data.

### Features:
1. Initialize connection to MT5 terminal.
2. Query tick range for a specific day (default: yesterday).
3. Preprocess tick data into a pandas DataFrame.
4. Calculate TPO distribution across customizable price bins (`TPO_BIN_PIPS`) and time brackets (`TIME_BRACKET`).
5. Compute **Point of Control (POC)** and **Value Area (VAH / VAL)** (~70% TPO range).
6. Visualize side-by-side Plotly chart: Price Action & Indicator Lines (Left) + Horizontal TPO Histogram Profile (Right).


In [8]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, UTC
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set pandas options for better display
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

In [9]:
# Initialize MetaTrader 5 connection
if not mt5.initialize():
    print("MetaTrader5 initialization failed, error code:", mt5.last_error())
    quit()
else:
    print("MetaTrader5 initialized successfully.")

    # Print terminal connection info
    info = mt5.terminal_info()
    print(f"Connected to: {info.company} - {info.name}")
    print(f"Version: {mt5.version()}")

MetaTrader5 initialized successfully.
Connected to: RoboForex Ltd - RoboForex MT5 Terminal
Version: (500, 6090, '31 Jul 2026')


## Define Parameters

By default, we fetch ticks for the `EURUSD` symbol for **yesterday** (the day prior to current execution).
You can customize the `SYMBOL` and date range below.

In [10]:
# --- Configuration ---
SYMBOL = "USDJPY"  # Symbol to fetch
today = datetime.now()
yesterday = today - timedelta(days=1)

# Start and end of yesterday
date_from = datetime(yesterday.year, yesterday.month, yesterday.day, 0, 0, 0, tzinfo=UTC)
date_to = datetime(yesterday.year, yesterday.month, yesterday.day, 23, 59, 59, tzinfo=UTC)

print(f"Symbol: {SYMBOL}")
print(f"Date From: {date_from}")
print(f"Date To:   {date_to}")

Symbol: USDJPY
Date From: 2026-08-12 00:00:00+00:00
Date To:   2026-08-12 23:59:59+00:00


## Fetch Tick Data

We use `mt5.copy_ticks_range` to request tick data. The flag `mt5.COPY_TICKS_ALL` retrieves all tick types (bid, ask, etc.).

In [11]:
print(f"Requesting tick data for {SYMBOL}...")
ticks = mt5.copy_ticks_range(SYMBOL, date_from, date_to, mt5.COPY_TICKS_ALL)

if ticks is None or len(ticks) == 0:
    print(f"No ticks retrieved. Check if {SYMBOL} is available in Market Watch or if the market was open during the selected range.")
    print("Error code:", mt5.last_error())
else:
    print(f"Successfully retrieved {len(ticks):,} ticks.")

Requesting tick data for USDJPY...
Successfully retrieved 75,499 ticks.


## Preprocess Data

Let's convert the fetched ticks array into a pandas DataFrame, format the timestamps, and compute the raw spread (`ask - bid`).

To make the calculation universal, we fetch the symbol's information using `mt5.symbol_info` to get its `point` size and `digits` count. A standard **pip** is typically defined as 10 points for currency pairs with 3 or 5 decimal digits, and as 1 point for other symbols (such as gold, crypto, or indices).

In [12]:
if ticks is not None and len(ticks) > 0:
    # Create DataFrame
    df = pd.DataFrame(ticks)

    # Convert millisecond timestamp to pandas datetime
    df['time'] = pd.to_datetime(df['time_msc'], unit='ms')

    # Set the time column as the index for easier analysis
    df.set_index('time', inplace=True)

    # Fetch instrument info for universal pip calculation
    info = mt5.symbol_info(SYMBOL)
    if info is not None:
        point = info.point
        # A standard pip is 10 points for 3/5 digit forex pairs, and 1 point for others
        if info.digits in [3, 5]:
            pip_size = 10 * point
        else:
            pip_size = point
        print(f"Universal Pip Calculation: 1 Pip = {pip_size} (Point: {point}, Digits: {info.digits})")
    else:
        pip_size = 0.0001
        print(f"Symbol info not found for {SYMBOL}. Using fallback 1 Pip = {pip_size}")

    # Calculate bid-ask spread
    df['spread_raw'] = df['ask'] - df['bid']
    df['spread_pips'] = df['spread_raw'] / pip_size

    # Display the first few rows
    print("\nPreprocessed Tick DataFrame:")
    display(df.head())
else:
    print("No data to preprocess.")

Universal Pip Calculation: 1 Pip = 0.01 (Point: 0.001, Digits: 3)

Preprocessed Tick DataFrame:


,bid,ask,last,volume,time_msc,flags,volume_real,spread_raw,spread_pips
time,,,,,,,,,
2026-08-12 00:05:02.221,159.224,159.389,0.0,0,1786493102221,134,0.0,0.165,16.5
2026-08-12 00:05:02.285,159.222,159.389,0.0,0,1786493102285,130,0.0,0.167,16.7
2026-08-12 00:05:02.381,159.217,159.389,0.0,0,1786493102381,130,0.0,0.172,17.2
2026-08-12 00:06:49.036,159.224,159.358,0.0,0,1786493209036,134,0.0,0.134,13.4
2026-08-12 00:06:49.132,159.214,159.331,0.0,0,1786493209132,134,0.0,0.117,11.7


## Summary Statistics

Let's look at the basic statistics of the ticks, such as average, maximum, and minimum spreads.

In [13]:
if ticks is not None and len(ticks) > 0:
    print("--- Summary Statistics ---")
    print(f"Total Ticks: {len(df):,}")
    print(f"Min Bid: {df['bid'].min():.5f}")
    print(f"Max Ask: {df['ask'].max():.5f}")
    print(f"Average Spread (pips): {df['spread_pips'].mean():.2f}")
    print(f"Max Spread (pips): {df['spread_pips'].max():.2f}")
    print(f"Min Spread (pips): {df['spread_pips'].min():.2f}")
else:
    print("No data available for statistics.")

--- Summary Statistics ---
Total Ticks: 75,499
Min Bid: 158.57400
Max Ask: 159.54600
Average Spread (pips): 0.39
Max Spread (pips): 17.20
Min Spread (pips): 0.00


## TPO (Time Price Opportunity) Market Profile Analysis

We resample tick data to calculate the **TPO Market Profile** distribution for the session, identifying key auction market levels:
- **Point of Control (POC)**: The price level with the highest density of TPOs (highlighted in gold).
- **Value Area (VAH / VAL)**: The price range containing **~70%** of the session's TPOs (highlighted in blue).

You can customize `TPO_BIN_PIPS` (price resolution in pips) and `TIME_BRACKET` (bracket resolution: e.g., `'30min'`, `'15min'`, `'1h'`) below.

In [14]:
# --- TPO Market Profile Configuration ---
TPO_BIN_PIPS = 0.5       # Price bin size in pips (e.g., 0.5, 1.0)
TIME_BRACKET = "30min"   # Time bracket interval: '30min', '15min', '1h', etc.

if ticks is not None and len(ticks) > 0:
    print(f"Calculating TPO Market Profile ({TIME_BRACKET} brackets, {TPO_BIN_PIPS} pip bins)...")

    price_min = df['bid'].min()
    price_max = df['ask'].max()
    bin_size = TPO_BIN_PIPS * pip_size
    if bin_size <= 0:
        bin_size = 0.00005

    price_bins = np.arange(price_min, price_max + bin_size, bin_size)
    bin_centers = (price_bins[:-1] + price_bins[1:]) / 2.0

    # Resample tick data into time brackets
    brackets = df['bid'].resample(TIME_BRACKET).agg(['min', 'max']).dropna()

    # Count TPOs per price bin
    tpo_counts = np.zeros(len(price_bins) - 1, dtype=int)
    for _, row in brackets.iterrows():
        mask = (price_bins[:-1] >= row['min'] - bin_size/2) & (price_bins[1:] <= row['max'] + bin_size/2)
        tpo_counts += mask.astype(int)

    # Point of Control (POC)
    poc_idx = int(np.argmax(tpo_counts))
    poc_price = bin_centers[poc_idx]

    # Value Area (~70% of TPOs)
    total_tpos = tpo_counts.sum()
    target_tpos = total_tpos * 0.70
    sorted_indices = np.argsort(tpo_counts)[::-1]

    cum_tpos = 0
    va_mask = np.zeros(len(tpo_counts), dtype=bool)
    for i in sorted_indices:
        va_mask[i] = True
        cum_tpos += tpo_counts[i]
        if cum_tpos >= target_tpos:
            break

    va_prices = bin_centers[va_mask]
    val_price = float(va_prices.min()) if len(va_prices) > 0 else poc_price
    vah_price = float(va_prices.max()) if len(va_prices) > 0 else poc_price

    print("--- TPO Market Profile Statistics ---")
    print(f"Total Time Brackets ({TIME_BRACKET}): {len(brackets)}")
    print(f"Total TPOs: {total_tpos:,}")
    print(f"Point of Control (POC): {poc_price:.5f}")
    print(f"Value Area High (VAH):  {vah_price:.5f}")
    print(f"Value Area Low (VAL):   {val_price:.5f}")

    # Create Side-by-Side Figure: Left = Price Line & Key Levels, Right = TPO Profile Histogram
    fig_tpo = make_subplots(
        rows=1, cols=2,
        column_widths=[0.7, 0.3],
        shared_yaxes=True,
        horizontal_spacing=0.03,
        subplot_titles=(f"{SYMBOL} Price Action & Key TPO Levels", "TPO Profile Distribution")
    )

    # Left Subplot: Bid Price Line
    fig_tpo.add_trace(
        go.Scatter(
            x=df.index,
            y=df['bid'],
            mode='lines',
            name='Bid',
            line=dict(color='#1f77b4', width=1.2),
            hovertemplate="<b>Time:</b> %{x|%Y-%m-%d %H:%M:%S}<br><b>Bid:</b> %{y:.5f}<extra>Bid</extra>"
        ),
        row=1, col=1
    )

    # Key Level Horizontal Lines
    fig_tpo.add_hline(
        y=poc_price,
        line=dict(color='#ffc107', width=2.5, dash='solid'),
        annotation_text=f"POC ({poc_price:.5f})",
        annotation_position="top left",
        row=1, col=1
    )
    fig_tpo.add_hline(
        y=vah_price,
        line=dict(color='#dc3545', width=1.8, dash='dash'),
        annotation_text=f"VAH ({vah_price:.5f})",
        annotation_position="top left",
        row=1, col=1
    )
    fig_tpo.add_hline(
        y=val_price,
        line=dict(color='#28a745', width=1.8, dash='dash'),
        annotation_text=f"VAL ({val_price:.5f})",
        annotation_position="bottom left",
        row=1, col=1
    )

    # Right Subplot: Horizontal TPO Profile Histogram
    colors = ['#ffc107' if i == poc_idx else ('#4682b4' if is_va else '#d0d7de') for i, is_va in enumerate(va_mask)]
    fig_tpo.add_trace(
        go.Bar(
            y=bin_centers,
            x=tpo_counts,
            orientation='h',
            name='TPOs',
            marker_color=colors,
            hovertemplate="<b>Price:</b> %{y:.5f}<br><b>TPOs:</b> %{x}<extra>TPO</extra>"
        ),
        row=1, col=2
    )

    # Layout Setup
    date_str = df.index[0].strftime('%Y-%m-%d') if len(df) > 0 else ""
    fig_tpo.update_layout(
        title=dict(
            text=f"MetaTrader 5 - {SYMBOL} Interactive TPO Market Profile ({date_str})",
            x=0.5,
            xanchor='center'
        ),
        height=750,
        template='plotly_white',
        showlegend=False,
        margin=dict(l=60, r=40, t=100, b=60)
    )

    fig_tpo.update_yaxes(title_text="Price (Bid)", row=1, col=1)
    fig_tpo.update_xaxes(title_text="TPO Count", row=1, col=2)

    fig_tpo.show()
else:
    print("No tick data available for TPO profile analysis.")

Calculating TPO Market Profile (30min brackets, 0.5 pip bins)...
--- TPO Market Profile Statistics ---
Total Time Brackets (30min): 48
Total TPOs: 1,014
Point of Control (POC): 159.36650
Value Area High (VAH):  159.49650
Value Area Low (VAL):   159.01150
